# Streaming Subscription Churn and User Engagement

**Contributors:** [Mason DeWitt]("https://github.com/madewitt55), [Elena Van]("https://github.com/3lenavan), [Prithvi Ghale]("https://github.com/PrithviGhale)

Subscription-based streaming platforms depend on recurring revenue and sustained engagement. When
activity declines, subscriptions are paused, or customers encounter billing and service problems, users
may be more likely to cancel. In this project, our team builds multiple classification model to predict
subscription churn from customer, payment, and engagement behavior. The models support targeted
retention strategies such as content discovery, service recovery, or plan adjustments.

## Datasets
The data for this project is provided by Kaggle. Their [Streaming Subscription Churn Model Competiton]("https://www.kaggle.com/competitions/streaming-subscription-churn-model) provides separate training and testing datasets. The data is synthetic; it should not be used to draw conclusions about real streaming service users and their behavior. Kaggle provides this data purely for student evaluation and it should not be used to train commerical models or embedded within paid software products. The rights to this data remain restricted to the competition's context.

In [ ]:
import pandas as pd

train_df = pd.read_csv("datasets/train.csv")
test_df = pd.read_csv("datasets/test.csv")

print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")

Train: (125000, 20)
Test: (75000, 19)


The training data set consists of **125,000 records** and **20 variables**. The testing set consists of **75,000 records** and **19 variables** (the target variable is excluded from the testing set).

## Data Quality Analysis

### Variables

Kaggle's datasets provide 20 total variables along with descriptive documentation defining units wherever necessary.

#### Identifiers
- `customer_id` - A unique customer identification number

Indentifiers provide no real description of the customer and are excluded from modeling.

#### Categorical Predictors
- `location` - The US state of the user
- `subscription_type` - Type of subscription
- `payment_plan` - How often the user pays, monthly or annually
- `payment_method` - Form of user payment
- `customer_service_inquiries` - The frequency of customer service inquiries from the user

Categorical predictors hold a non-numerical value and must be encoded before modeling.

#### Numerical Predictors
- `age` - The age of the user
- `num_subscription_pauses` - Number of times the user has paused their subscription (max 2)
- `signup_date` - Date the user signed up for the music subscription service
- `weekly_hours` - Average number of weekly listening hours
- `average_session_length` - Average length of each music listening session (in hours)
- `song_skip_rate` - Percentage of songs the user does not finish
- `weekly_songs_played` - Average number of songs the user plays in a week
- `weekly_unique_songs` - Average number of unique songs the user plays in a week
- `num_favorite_artists` - Number of artists the user set as favorite artists
- `num_platform_friends` - Number of user connections in the app
- `num_playlists_created` - Number of playlists the user created
- `num_shared_playlists` - Number of playlists that are shared publicly
- `notifications_clicked` - Number of in-app notifications clicked on

Numerical predictors hold a numerical value and do not require encoding before modeling.

#### Target
- `churned` - Whether the customer is active or churned
    - Positive class (1): Customer is **churned**
    - Negative class (0): Customer is **active**

The target variable is the customer's attribute to be predicted. We aim to classify a customer as either churned or active.

### Duplicate Identifiers

In [8]:
duplicate_check = pd.DataFrame({
    "duplicate_customer_ids": [
        train_df["customer_id"].duplicated().sum(),
        test_df["customer_id"].duplicated().sum(),
    ],
    "ids_shared_with_other_set": [
        train_df["customer_id"].isin(test_df["customer_id"]).sum(),
        test_df["customer_id"].isin(train_df["customer_id"]).sum(),
    ],
}, index=["train", "test"])
display(duplicate_check)

,duplicate_customer_ids,ids_shared_with_other_set
train,0,0
test,0,0


Neither the training nor testing set include duplicate values for a customer's unique identifier. No unique identifier is present in both sets.

### Missing Values

In [ ]:
# Count nulls plus empty or whitespace-only strings per column
def count_missing(df):
    blanks = df.select_dtypes("object").apply(lambda s: s.str.strip() == "").sum()
    return df.isna().sum().add(blanks, fill_value=0)


missing_values = pd.DataFrame({
    "train": count_missing(train_df),
    "test": count_missing(test_df),
}).astype("Int64")
display(missing_values)
print(f"Total missing - train: {missing_values['train'].sum()}, test: {missing_values['test'].sum()}")

,train,test
age,0,0
average_session_length,0,0
churned,0,<NA>
customer_id,0,0
customer_service_inquiries,0,0
location,0,0
notifications_clicked,0,0
num_favorite_artists,0,0
num_platform_friends,0,0
num_playlists_created,0,0


Total missing - train: 0, test: 0


Neither the training nor testing set include rows with missing variable values.

### Plausible Values

Plausible ranges are set from domain knowledge and documentation provided by Kaggle, not from the observed data.

| Variable | Plausible range | Rationale |
|---|---|---|
| `customer_id` | Integer >= 1 | Positive identifier |
| `age` | 13 to 100 | Typical minimum age for a streaming account |
| `num_subscription_pauses` | 0 to 2 | Documented maximum of 2 |
| `signup_date` | -3650 to 0 | Days relative to a reference date, up to 10 years back, never in the future |
| `weekly_hours` | 0 to 168 | Hours in a week |
| `average_session_length` | 0 to 24 | 24 hour max listening session |
| `song_skip_rate` | 0 to 1 | Stored as a proportion |
| `weekly_songs_played` | 0 to 3500 | About 20 songs per hour over 168 hours |
| `weekly_unique_songs` | 0 to 3500, and <= `weekly_songs_played` | Cannot play more unique songs than total songs |
| `num_favorite_artists` | 0 to 1000 | Non-negative count with a generous cap |
| `num_platform_friends` | 0 to 5000 | Non-negative count with a generous cap |
| `num_playlists_created` | 0 to 1000 | Non-negative count with a generous cap |
| `num_shared_playlists` | 0 to 1000 | Non-negative count with a generous cap |
| `notifications_clicked` | 0 to 1000 | Non-negative count with a generous cap |
| `churned` | 0 or 1 | Binary target |
| `location` | One of the 50 US states | Spelled correctly |
| `subscription_type` | Free, Premium, Student, Family | Known plan types |
| `payment_plan` | Monthly, Yearly | Known billing options |
| `payment_method` | Credit Card, Debit Card, Paypal, Apple Pay | Known payment methods |
| `customer_service_inquiries` | Low, Medium, High | Known frequency levels |

In [26]:
numeric_ranges = {
    "age": (13, 100),
    "num_subscription_pauses": (0, 2),
    "signup_date": (-3650, 0),
    "weekly_hours": (0, 168),
    "average_session_length": (0, 24),
    "song_skip_rate": (0, 1),
    "weekly_songs_played": (0, 3500),
    "weekly_unique_songs": (0, 3500),
    "num_favorite_artists": (0, 1000),
    "num_platform_friends": (0, 5000),
    "num_playlists_created": (0, 1000),
    "num_shared_playlists": (0, 1000),
    "notifications_clicked": (0, 1000),
}

allowed_categories = {
    "location": {
        "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut",
        "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
        "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan",
        "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire",
        "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio",
        "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
        "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia",
        "Wisconsin", "Wyoming",
    },
    "subscription_type": {"Free", "Premium", "Student", "Family"},
    "payment_plan": {"Monthly", "Yearly"},
    "payment_method": {"Credit Card", "Debit Card", "Paypal", "Apple Pay"},
    "customer_service_inquiries": {"Low", "Medium", "High"},
}

# Count rows outside the plausible range for each variable
def count_violations(df):
    counts = {"customer_id": int((df["customer_id"] < 1).sum())}
    for col, (low, high) in numeric_ranges.items():
        counts[col] = int((~df[col].between(low, high)).sum())
    for col, allowed in allowed_categories.items():
        counts[col] = int((~df[col].isin(allowed)).sum())
    if "churned" in df:
        counts["churned"] = int((~df["churned"].isin([0, 1])).sum())
    counts["weekly_unique_songs > weekly_songs_played"] = int(
        (df["weekly_unique_songs"] > df["weekly_songs_played"]).sum()
    )
    return pd.Series(counts)


range_violations = pd.DataFrame({
    "train": count_violations(train_df),
    "test": count_violations(test_df),
}).astype("Int64")  # test has no churned column (<NA>)
display(range_violations)

all_in_range = (range_violations.fillna(0) == 0).all().all()
print(f"All values plausible: {all_in_range}")

,train,test
age,0,0
average_session_length,100768,60497
churned,0,<NA>
customer_id,0,0
customer_service_inquiries,0,0
location,6601,4026
notifications_clicked,0,0
num_favorite_artists,0,0
num_platform_friends,0,0
num_playlists_created,0,0


All values plausible: False


#### Session Length Anomalies

The plausible value check saw *161,265* total out-of-range values for numerical predictor `average_session_length`. Kaggle's documentation of these datasets defines the units as hours.

In [27]:
session_min, session_max = numeric_ranges["average_session_length"]

session_summary = pd.DataFrame({
    "observed_min": [df["average_session_length"].min() for df in (train_df, test_df)],
    "observed_max": [df["average_session_length"].max() for df in (train_df, test_df)],
    "pct_above_range": [(df["average_session_length"] > session_max).mean() * 100 for df in (train_df, test_df)],
    "pct_below_range": [(df["average_session_length"] < session_min).mean() * 100 for df in (train_df, test_df)],
}, index=["train", "test"])
session_summary.insert(0, "plausible_range", f"{session_min} to {session_max}")
display(session_summary.round(2))

,plausible_range,observed_min,observed_max,pct_above_range,pct_below_range
train,0 to 24,1.0,120.0,80.61,0.0
test,0 to 24,1.0,120.0,80.66,0.0


The observed maximum of this variable in both the training and testing sets is *120.* With the units as hours (defined by Kaggle), this represents 5 straight days of listening. With the units as minutes, this represents 2 hours of listening (much more sensible). Since over 80% of all rows hold a value greater than 24 for `average_session_length`, it is highly likely that variable's units were intended to be documented as minutes. We will adjust our units and plausible value range accordingly. 

#### Location Anomalies

The plausible value check saw *10,627* total invalid values for categorical predictor `location`. Kaggle's documentation of the data states this variable should hold a string value of a valid US state.

In [29]:
import difflib

valid_states = allowed_categories["location"]

location_counts = pd.DataFrame({
    "train": train_df["location"].value_counts(),
    "test": test_df["location"].value_counts(),
}).fillna(0).astype(int)
location_counts["total"] = location_counts["train"] + location_counts["test"]
location_counts["valid_state"] = location_counts.index.isin(valid_states)

# for each invalid label, find the closest valid state and check whether that state appears in the data
invalid_locations = location_counts[~location_counts["valid_state"]].copy()
invalid_locations["closest_valid_state"] = [
    next(iter(difflib.get_close_matches(loc, valid_states, n=1, cutoff=0.8)), None)
    for loc in invalid_locations.index
]
display(invalid_locations)

,train,test,total,valid_state,closest_valid_state
location,,,,,
Nebrasksa,6601,4026,10627,False,Nebraska


In [ ]:
# "Nebrasksa" is the only invalid location and has no correctly spelled counterpart in the data,
# so it is treated as a misspelling of Nebraska. Only the dataframes are changed, not the CSVs.
for name, df in (("train", train_df), ("test", test_df)):
    typo_rows = (df["location"] == "Nebrasksa").sum()
    df["location"] = df["location"].replace("Nebrasksa", "Nebraska")
    print(f"{name}: corrected {typo_rows} rows, invalid locations remaining: "
          f"{(~df['location'].isin(valid_states)).sum()}")

The only non-valid location the appears in both the training and testing sets is *"Nebrasksa."* It is highly likely that this is simply a frequent typo meant to represent the actual valid US state *Nebraska*. We will modify these rows to hold the correct and intended state.

### Subscription Pauses Anomaly
The plausible value check saw *49694* out-of-range values for numerical predictor `num_subscriptions_pauses`. Kaggle's documentation of these datasets defines the maximum value as 2.

In [13]:
pause_min, pause_max = numeric_ranges["num_subscription_pauses"]

pause_summary = pd.DataFrame({
    "observed_min": [df["num_subscription_pauses"].min() for df in (train_df, test_df)],
    "observed_max": [df["num_subscription_pauses"].max() for df in (train_df, test_df)],
    "pct_above_max": [(df["num_subscription_pauses"] > pause_max).mean() * 100 for df in (train_df, test_df)],
    "pct_below_min": [(df["num_subscription_pauses"] < pause_min).mean() * 100 for df in (train_df, test_df)],
}, index=["train", "test"])
pause_summary.insert(0, "documented_range", f"{pause_min} to {pause_max}")
display(pause_summary.round(2))

,documented_range,observed_min,observed_max,pct_above_max,pct_below_min
train,0 to 2,0,4,39.76,0.0
test,0 to 2,0,4,39.79,0.0


#### Weekly Unique Songs vs Weekly Total Songs Anomalies

The plausible value check saw *59,424* invalid values for numerical predictor `weekly_unique_songs`. Kaggle's documentation of these datasets defines this variable as the average number of unique songs a user plays in a week, which cannot exceed the total number of songs played (`weekly_songs_played`).

##### Option 1: KNN Imputation

Rows where `weekly_unique_songs` exceeds `weekly_songs_played` could have one of the two values marked as missing and replaced with the average value of the most similar customers, as measured by their other predictors. Either variable could be imputed, since it is not known which of the two is wrong. This only works if the other predictors carry information about the variable being imputed.

In [44]:
song_cols = ["weekly_unique_songs", "weekly_songs_played"]

song_corr = train_df[list(numeric_ranges)].corr()[song_cols]

# order predictors by their strongest correlation with another variable (self-correlation of 1 excluded)
off_diagonal = song_corr.copy()
for col in song_cols:
    off_diagonal.loc[col, col] = float("nan")
song_corr = song_corr.loc[off_diagonal.abs().max(axis=1).sort_values(ascending=False).index]
display(song_corr.add_prefix("corr_with_").round(3))

,corr_with_weekly_unique_songs,corr_with_weekly_songs_played
num_favorite_artists,0.005,0.005
average_session_length,-0.005,-0.000
num_platform_friends,-0.001,0.003
song_skip_rate,0.001,-0.003
notifications_clicked,0.003,-0.003
signup_date,0.000,0.003
num_subscription_pauses,-0.003,-0.001
num_playlists_created,0.000,-0.002
num_shared_playlists,-0.000,0.002
age,-0.001,-0.002


The Linear (Pearson) correlations between `weekly_unique_songs`

### Outlier Detection

### Parse Date Fields

### Encode Categorical Variables

### Predictor vs Predictor Correlation

### Predictor Class Distribution

### Data Quality Analysis Conclusion

## Feature Engineering

## Train Classification Models

### Scale Predictors

### Logstic Regression

### K-Nearest Neighbors Classifier

### Weighted K-Nearest Neighbors Classifier

### Random Forest Classifier

## Model Evaluation

## Retention for At-Risk Subscribers